# Imports

In [69]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# EDA

In [70]:
train = pd.read_csv("Datasets/train.csv")
test = pd.read_csv("Datasets/test.csv")

print("Training shape:", train.shape)
print("Test shape:", test.shape)

display(train.head())

Training shape: (891, 12)
Test shape: (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [71]:
print("\nTraining information:")
train.info()

print("\nMissing values:")
print(train.isnull().sum())

print("\nSurvival distribution:")
print(train["Survived"].value_counts())

print("\nSurvival rate:")
print(train["Survived"].mean())


Training information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB

Missing values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin         

In [72]:
def engineer_features(df):
    df = df.copy()

    # Title

    df["Title"] = df["Name"].str.extract(
        r",\s*([^.]*)\."
    )

    df["Title"] = df["Title"].replace({
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs"
    })

    common_titles = [
        "Mr",
        "Miss",
        "Mrs",
        "Master"
    ]

    df["Title"] = df["Title"].where(
        df["Title"].isin(common_titles),
        "Rare"
    )

    # Family size

    df["FamilySize"] = (
        df["SibSp"] +
        df["Parch"] +
        1
    )

    df["IsAlone"] = (
        df["FamilySize"] == 1
    ).astype(int)

    # Family size category
    df["FamilyCategory"] = pd.cut(
        df["FamilySize"],
        bins=[0, 1, 4, 7, 20],
        labels=[
            "Alone",
            "Small",
            "Medium",
            "Large"
        ]
    )

    # Ticket group

    df["TicketGroupSize"] = (
        df["Ticket"]
        .map(df["Ticket"].value_counts())
    )

    # Fare per person
    df["FarePerPerson"] = (
        df["Fare"] /
        df["TicketGroupSize"].replace(0, np.nan)
    )

    # Child

    df["Child"] = (
        df["Age"] < 16
    ).astype(int)

    # Woman or child

    df["WomanOrChild"] = (
        (df["Sex"] == "female") |
        (df["Age"] < 16)
    ).astype(int)

    # Cabin information

    df["HasCabin"] = (
        df["Cabin"].notna()
    ).astype(int)

    df["Deck"] = (
        df["Cabin"]
        .fillna("U")
        .str[0]
    )

    # Surname

    df["Surname"] = (
        df["Name"]
        .str.split(",")
        .str[0]
    )

    # Sex × Class interaction

    df["SexClass"] = (
        df["Sex"] +
        "_" +
        df["Pclass"].astype(str)
    )

    # Age imputation based on Title + Pclass

    age_group_median = (
        df.groupby(
            ["Title", "Pclass"],
            observed=True
        )["Age"]
        .transform("median")
    )

    df["Age"] = df["Age"].fillna(age_group_median)

    # Final fallback
    df["Age"] = df["Age"].fillna(
        df["Age"].median()
    )

    # Recalculate age-derived features after imputation
    df["Child"] = (
        df["Age"] < 16
    ).astype(int)

    df["WomanOrChild"] = (
        (df["Sex"] == "female") |
        (df["Age"] < 16)
    ).astype(int)

    return df


train_fe = engineer_features(train)
test_fe = engineer_features(test)

In [73]:
def add_group_survival_features(train_part, other_part):
    """
    Calculate survival statistics using train_part only.

    train_part:
        Data containing known Survived values.

    other_part:
        Data for which we want to generate features.
        This can be validation data or Kaggle test data.
    """

    train_part = train_part.copy()
    other_part = other_part.copy()

    # Ticket survival

    ticket_stats = (
        train_part
        .groupby("Ticket")["Survived"]
        .agg(["sum", "count"])
    )

    ticket_sum = train_part["Ticket"].map(
        ticket_stats["sum"]
    )

    ticket_count = train_part["Ticket"].map(
        ticket_stats["count"]
    )

    train_part["GroupSurvival"] = (
        (ticket_sum - train_part["Survived"]) /
        (ticket_count - 1)
    )

    train_part["GroupSurvival"] = (
        train_part["GroupSurvival"]
        .fillna(0.5)
    )

    other_part["GroupSurvival"] = (
        other_part["Ticket"].map(
            ticket_stats["sum"]
        )
        /
        other_part["Ticket"].map(
            ticket_stats["count"]
        )
    )

    other_part["GroupSurvival"] = (
        other_part["GroupSurvival"]
        .fillna(0.5)
    )

    # Family + ticket survival

    train_part["FamilyTicket"] = (
        train_part["Surname"] +
        "_" +
        train_part["Ticket"].astype(str)
    )

    other_part["FamilyTicket"] = (
        other_part["Surname"] +
        "_" +
        other_part["Ticket"].astype(str)
    )

    family_stats = (
        train_part
        .groupby("FamilyTicket")["Survived"]
        .agg(["sum", "count"])
    )

    family_sum = train_part["FamilyTicket"].map(
        family_stats["sum"]
    )

    family_count = train_part["FamilyTicket"].map(
        family_stats["count"]
    )

    train_part["FamilyTicketSurvival"] = (
        (family_sum - train_part["Survived"]) /
        (family_count - 1)
    )

    train_part["FamilyTicketSurvival"] = (
        train_part["FamilyTicketSurvival"]
        .fillna(0.5)
    )

    other_part["FamilyTicketSurvival"] = (
        other_part["FamilyTicket"].map(
            family_stats["sum"]
        )
        /
        other_part["FamilyTicket"].map(
            family_stats["count"]
        )
    )

    other_part["FamilyTicketSurvival"] = (
        other_part["FamilyTicketSurvival"]
        .fillna(0.5)
    )

    return train_part, other_part


In [74]:
train_indices, val_indices = train_test_split(
    np.arange(len(train_fe)),
    test_size=0.20,
    random_state=42,
    stratify=train_fe["Survived"]
)

train_part = train_fe.iloc[train_indices].copy()
val_part = train_fe.iloc[val_indices].copy()

print("\nTraining rows:", len(train_part))
print("Validation rows:", len(val_part))


Training rows: 712
Validation rows: 179


In [75]:
train_part, val_part = add_group_survival_features(
    train_part,
    val_part
)

In [76]:
features = [
    "Pclass",
    "Sex",
    "Age",
    "Fare",
    "SibSp",
    "Parch",
    "Embarked",

    "Title",

    "FamilySize",
    "IsAlone",
    "FamilyCategory",

    "FarePerPerson",

    "Child",
    "WomanOrChild",

    "HasCabin",
    "Deck",

    "TicketGroupSize",

    "GroupSurvival",
    "FamilyTicketSurvival",

    "SexClass"
]

In [77]:
numeric_features = [
    "Pclass",
    "Age",
    "Fare",
    "SibSp",
    "Parch",
    "FamilySize",
    "IsAlone",
    "FarePerPerson",
    "Child",
    "WomanOrChild",
    "HasCabin",
    "TicketGroupSize",
    "GroupSurvival",
    "FamilyTicketSurvival"
]

categorical_features = [
    "Sex",
    "Embarked",
    "Title",
    "FamilyCategory",
    "Deck",
    "SexClass"
]


In [78]:
X_train = train_part[features]
y_train = train_part["Survived"]

X_val = val_part[features]
y_val = val_part["Survived"]

# Model

In [79]:
numeric_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    )
])

categorical_transformer = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="most_frequent")
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

preprocessor = ColumnTransformer([
    (
        "num",
        numeric_transformer,
        numeric_features
    ),
    (
        "cat",
        categorical_transformer,
        categorical_features
    )
])

In [80]:
model = RandomForestClassifier(
    n_estimators=500,
    max_depth=6,
    min_samples_split=8,
    min_samples_leaf=3,
    max_features="sqrt",
    random_state=42
)

In [81]:
pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        model
    )
])

In [82]:
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [83]:
val_predictions = pipeline.predict(
    X_val
)

accuracy = accuracy_score(
    y_val,
    val_predictions
)

print("\n================================")
print("VALIDATION RESULTS")
print("================================")
print(f"Accuracy: {accuracy:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_val,
        val_predictions
    )
)


VALIDATION RESULTS
Accuracy: 0.8045

Classification report:
              precision    recall  f1-score   support

           0       0.83      0.85      0.84       110
           1       0.76      0.72      0.74        69

    accuracy                           0.80       179
   macro avg       0.79      0.79      0.79       179
weighted avg       0.80      0.80      0.80       179



In [84]:
results = val_part[
    [
        "PassengerId",
        "Survived",
        "Sex",
        "Pclass",
        "Age",
        "Title",
        "FamilySize"
    ]
].copy()

In [85]:
results["Prediction"] = val_predictions

results["Correct"] = (
    results["Survived"] ==
    results["Prediction"]
)

display(results.head(20))

,PassengerId,Survived,Sex,Pclass,Age,Title,FamilySize,Prediction,Correct
565,566,0,male,3,24.0,Mr,3,0,True
160,161,0,male,3,44.0,Mr,2,0,True
553,554,1,male,3,22.0,Mr,1,0,False
860,861,0,male,3,41.0,Mr,3,0,True
241,242,1,female,3,18.0,Miss,2,1,True
559,560,1,female,3,36.0,Mrs,2,1,True
387,388,1,female,2,36.0,Miss,1,1,True
536,537,0,male,1,45.0,Rare,1,1,False
698,699,0,male,1,49.0,Mr,3,1,False
99,100,0,male,2,34.0,Mr,2,0,True


# Final Use

In [86]:
full_train_fe, final_test_fe = (
    add_group_survival_features(
        train_fe,
        test_fe
    )
)

In [87]:
X_full = full_train_fe[features]
y_full = full_train_fe["Survived"]

X_test_final = final_test_fe[features]

In [88]:
final_pipeline = Pipeline([
    (
        "preprocessor",
        preprocessor
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=500,
            max_depth=6,
            min_samples_split=8,
            min_samples_leaf=3,
            max_features="sqrt",
            random_state=42
        )
    )
])

In [89]:
final_pipeline.fit(
    X_full,
    y_full
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers co

In [90]:
test_predictions = final_pipeline.predict(
    X_test_final
)

In [91]:
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": test_predictions
})

In [92]:
print("\n================================")
print("SUBMISSION")
print("================================")

display(submission.head(10))

print("\nPrediction counts:")
print(
    submission["Survived"].value_counts()
)


SUBMISSION


,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0
8,900,1
9,901,0



Prediction counts:
Survived
0    257
1    161
Name: count, dtype: int64


In [93]:
submission.to_csv(
    "submission.csv",
    index=False
)

print("\nSaved: submission.csv")


Saved: submission.csv
